In [ ]:
import os
import logging
import torch
import torch.distributed as dist
import numpy as np
import cv2
from PIL import Image
from tqdm import tqdm
import torch

from diffusers import DiffusionPipeline
from diffusers.utils import load_image, export_to_video

from wan.configs import WAN_CONFIGS, SIZE_CONFIGS
from wan.vace import WanVace
from wan.utils.utils import cache_video

logging.basicConfig(level=logging.INFO)
device = 'cuda' if torch.accelerator.is_available() else 'gpu'
OFFLOAD_MODEL = True

In [ ]:
TASK = "vace-1.3B"
CKPT_DIR = "D:/Desktop/xqy/NUS311/FYP/models/Wan2.1-VACE-1.3B"
REF_IMG_PATH = "D:/Desktop/xqy/NUS311/FYP/project/Ego2ExowithMotion/data/example/ref.png"
EGO_VIDEO_PATH = "D:/Desktop/xqy/NUS311/FYP/project/Ego2ExowithMotion/data/example/ego.mp4"
OUTPUT_PATH = "D:/Desktop/xqy/NUS311/FYP/project/Ego2ExowithMotion/data/example/output/tpv.mp4"
SIZE_KEY = "832*480"
FRAME_NUM = 300
PROMPT = "Third-person view, following camera shot from behind. "\
         "capturing the full body and movement. "\
         "High quality, cinematic lighting, 4k, 3d game engine style."  

In [ ]:
logging.info(f"Initializing Wan-VACE pipeline for task: {TASK}")
if TASK not in WAN_CONFIGS:
    raise ValueError(f"Task {TASK} not found in configs.")
cfg = WAN_CONFIGS[TASK]

In [ ]:
def generate_full_mask(video_path, size):
    """
    生成全屏 Mask 视频 (全白 MP4)，用于指示模型重新生成整个画面。
    必须生成视频文件，因为 WanVace 的底层 decord 不支持读取单张图片。
    """
    width, height = size
    
    # 1. 读取原视频的属性 (FPS 和 帧数)
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"无法打开视频文件: {video_path}")
        
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    
    # 2. 设置 Mask 视频路径 (改为 .mp4)
    mask_dir = os.path.dirname(video_path)
    mask_path = os.path.join(mask_dir, "temp_full_mask.mp4")
    
    # 3. 创建视频写入器
    # 使用 mp4v 编码器生成 mp4 文件
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(mask_path, fourcc, fps, (width, height), isColor=True)
    
    # 4. 生成全白帧并写入
    # 255 代表全白 (Reactive区域，即生成区域)
    # 注意：WanVace 内部会读取 RGB，所以我们需要 3 通道的白色
    white_frame = np.full((height, width, 3), 255, dtype=np.uint8)
    
    print(f"正在生成 Mask 视频 ({total_frames} 帧)...")
    for _ in range(total_frames):
        out.write(white_frame)
        
    out.release()
    
    print(f"Mask 视频已生成: {mask_path}")
    return mask_path

In [ ]:
def process_ref_image_center_crop(image_path, crop_ratio=1/3):
    """
    读取参考图，并只保留中间 1/3 区域 (以此作为人物特征)，其余部分变黑或裁剪。
    这里我们选择裁剪并Resize回原比例，或者填充黑色背景只留中间。
    策略：创建一个新图，只把原图中间部分贴进去，其余部分为黑色（或透明）。
    这能有效去除参考图背景对生成的干扰。
    """
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"Reference image not found: {image_path}")
    
    img = Image.open(image_path).convert("RGB")
    w, h = img.size
    
    # 计算中间区域
    center_w = w * crop_ratio
    left = (w - center_w) // 2
    right = left + center_w
    
    # 裁剪中间部分
    crop = img.crop((left, 0, right, h))
    
    # 策略：为了保持长宽比输入，我们将裁剪部分贴在一个黑色背景中心
    # 或者直接使用裁剪后的图（CLIP 会自动 Resize，但可能会变形）
    # 推荐：创建一个与原图等宽高的黑底图，将人物贴在中间
    new_img = Image.new("RGB", (w, h), (0, 0, 0))
    paste_x = int((w - crop.size[0]) // 2)
    new_img.paste(crop, (paste_x, 0))
    
    # 保存临时处理后的图片用于调试 (可选)
    processed_path = image_path.replace(".png", "_processed.png").replace(".jpg", "_processed.jpg")
    new_img.save(processed_path)
    logging.info(f"Processed Reference Image (Center 1/3) saved to: {processed_path}")
    
    return processed_path

Using `diffusers` to run inference.

In [ ]:
# pipe = DiffusionPipeline.from_pretrained(Wan-AI/Wan2.1-VACE-1.3B, dtype=torch.bfloat16, device_map="cuda")
# pipe.to("cuda")

# prompt = "A man with short gray hair plays a red electric guitar."
# image = load_image(
#     "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/guitar-man.png"
# )

# output = pipe(image=image, prompt=prompt).frames[0]
# export_to_video(output, "output.mp4")

Implement pipelines with given components(weight file).

In [ ]:
wan_vace = WanVace(
    config=cfg,
    checkpoint_dir=CKPT_DIR,
    device_id=0,
    rank=0,
    t5_fsdp=False,
    dit_fsdp=False,
    use_usp=False,
    t5_cpu=True,
)

In [ ]:
# logging.info("Preparing data sources...")
# proc_ref_path = process_ref_image_center_crop(REF_IMG_PATH)

In [ ]:
# B. 准备数据
logging.info("Generating Mask Video...")
# 生成 Mask 视频文件
mask_path = generate_full_mask(EGO_VIDEO_PATH, (target_w, target_h))

logging.info("Preparing VCU...")
# 注意 image_size 参数顺序：在 prepare_source 中通常期望 (H, W) 或者是根据内部 vid_proc 逻辑
# 查看源码，prepare_source 的 image_size 参数会被解包为 (h, w) = (size[1], size[0])
# 所以我们需要传入 (W, H) 格式的 SIZE_CONFIGS 值即可，内部会自动处理

src_video_t, src_mask_t, src_ref_images_t = wan_vace.prepare_source(
    src_video=[EGO_VIDEO_PATH],
    src_mask=[mask_path],             # 现在这是一个 .mp4 文件路径
    src_ref_images=[[REF_IMG_PATH]],  # 嵌套列表
    num_frames=FRAME_NUM,
    image_size=SIZE_CONFIGS[SIZE_KEY], # (W, H)
    device=wan_vace.device
)

In [ ]:
target_w, target_h = SIZE_CONFIGS[SIZE_KEY]
logging.info(f"Generating...")
video = wan_vace.generate(
    input_prompt=PROMPT,
    input_frames=src_video_t,
    input_masks=src_mask_t,
    input_ref_images=src_ref_images_t,
    size=(target_w, target_h),
    frame_num=FRAME_NUM,
    shift=5.0,
    sample_solver='unipc',
    sampling_steps=50,
    guide_scale=5.0,
    seed=42,
    offload_model=OFFLOAD_MODEL
)

In [ ]:
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
cache_video(
    tensor=video[None], 
    save_file=OUTPUT_PATH,
    fps=16,
    nrow=1,
    normalize=True,
    value_range=(-1, 1)
)
logging.info(f"Success! Video saved to: {OUTPUT_PATH}")

In [ ]:
# --- 配置参数 ---
# 模型路径 (请确保网络通畅或已下载到本地)
MODEL_ID = "Wan-AI/Wan2.1-VACE-1.3B"
SEED = 42  # 固定种子以便复现，设为 None 则随机
GUIDANCE_SCALE = 6.0  # 提示词相关性，较高通常能更好遵循 "3d game engine style"
HEIGHT = 720
WIDTH = 1280

pipe = DiffusionPipeline.from_pretrained(
    MODEL_ID, 
    dtype=torch.bfloat16,
    torch_dtype=torch.bfloat16
)
pipe.enable_model_cpu_offload()
pipe.enable_vae_slicing()

# --- 加载输入 ---
print("Loading reference image...")
image = load_image(REF_IMG_PATH)

# --- 设置随机种子 ---
if SEED is None:
    generator = None
else:
    generator = torch.Generator(device="cpu").manual_seed(SEED)

# --- 推理生成 ---
print("Generating video... (This may take a while)")
output = pipe(
    prompt=PROMPT,
    image=image,
    num_frames=FRAME_NUM,
    height=HEIGHT,
    width=WIDTH,
    guidance_scale=GUIDANCE_SCALE,
    generator=generator,
    num_inference_steps=30, # 步数越多质量通常越好，但速度越慢
).frames[0]

# --- 导出视频 ---
output_filename = "wan_vace_output_3rd_person.mp4"
print(f"Saving video to {output_filename}...")
export_to_video(output, output_filename, fps=60)
print("Done!")